# 2. Encryption and Hashing — Hands-on

The exam expects you to know the *difference* between encryption and hashing, the types of encryption, and why hashing alone isn't enough for passwords. This notebook lets you **do** each one.

## Key concepts

| | Encryption | Hashing |
|-|-----------|----------|
| **Purpose** | Keep data **secret** — only authorized parties can read it | Verify data **integrity** — detect if something changed |
| **Reversible?** | Yes (with the key) | No (one-way function) |
| **Output size** | Same size as input (approximately) | Fixed size regardless of input |
| **Use cases** | Data at rest, data in transit (TLS) | Passwords, file integrity, digital signatures |

In [ ]:
# We'll use Python's cryptography library — the same algorithms Azure uses
import hashlib
import os
import base64
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
import bcrypt

---
## 1. Symmetric encryption (AES)

**One key** encrypts and decrypts. Fast. Used for bulk data.

- Azure example: **Storage Service Encryption (SSE)** encrypts blobs at rest with AES-256.
- Real-world: HTTPS uses symmetric encryption for the data stream (after key exchange).

**Problem**: how do you safely share the key? That's where asymmetric encryption comes in.

In [ ]:
# Generate a random symmetric key
key = Fernet.generate_key()
print(f'Key (base64): {key.decode()}')
print(f'Key length:   {len(base64.urlsafe_b64decode(key)) * 8} bits\n')

# Encrypt
cipher = Fernet(key)
secret_message = b'Patient record: John Doe, DOB 1990-01-15, diagnosis: ...' 
encrypted = cipher.encrypt(secret_message)
print(f'Plaintext:  {secret_message.decode()}')
print(f'Encrypted:  {encrypted[:60]}...\n')

# Decrypt (same key)
decrypted = cipher.decrypt(encrypted)
print(f'Decrypted:  {decrypted.decode()}')
print(f'Match:      {decrypted == secret_message}')

In [ ]:
# What happens with the WRONG key?
wrong_key = Fernet.generate_key()
wrong_cipher = Fernet(wrong_key)
try:
    wrong_cipher.decrypt(encrypted)
except Exception as e:
    print(f'❌ Decryption with wrong key failed: {type(e).__name__}')
    print('   This is why key management (Azure Key Vault) matters!')

---
## 2. Asymmetric encryption (RSA)

**Two keys**: a public key (anyone can encrypt) and a private key (only the owner can decrypt). Slower, but solves the key-distribution problem.

- Azure example: **TLS handshake** — your browser uses the server's public key to securely send a symmetric session key.
- Azure example: **Azure Key Vault** can store RSA keys and perform sign/verify operations server-side.

In [ ]:
# Generate an RSA key pair
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()

# Anyone with the PUBLIC key can encrypt
message = b'Top secret: quarterly revenue is $42M'
encrypted = public_key.encrypt(
    message,
    padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
)
print(f'Plaintext:  {message.decode()}')
print(f'Encrypted:  {encrypted[:40].hex()}... ({len(encrypted)} bytes)')

# Only the PRIVATE key can decrypt
decrypted = private_key.decrypt(
    encrypted,
    padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
)
print(f'Decrypted:  {decrypted.decode()}')

### Symmetric vs Asymmetric — when to use which?

| | Symmetric | Asymmetric |
|-|-----------|------------|
| Speed | Fast | Slow (100-1000x slower) |
| Key distribution | Hard (must share secret key) | Easy (public key is... public) |
| Use case | Encrypting data in bulk | Key exchange, digital signatures, small payloads |
| Azure example | Storage encryption (AES-256) | TLS, Key Vault certificates |

In practice, **TLS combines both**: asymmetric for the handshake, symmetric for the data. Best of both worlds.

---
## 3. Hashing — one-way functions

A hash turns any input into a fixed-size fingerprint. You **cannot reverse it** — you can't get the original data from a hash.

- Azure example: **Entra ID stores password hashes**, not passwords.
- Use case: verifying file integrity (checksums).

In [ ]:
# Hash a message with SHA-256 (same algorithm Azure uses)
msg1 = b'Hello, world!'
msg2 = b'Hello, world.'  # one character different
msg3 = b'Hello, world!'  # identical to msg1

h1 = hashlib.sha256(msg1).hexdigest()
h2 = hashlib.sha256(msg2).hexdigest()
h3 = hashlib.sha256(msg3).hexdigest()

print(f'Message 1: "{msg1.decode()}"  → {h1}')
print(f'Message 2: "{msg2.decode()}"  → {h2}')
print(f'Message 3: "{msg3.decode()}"  → {h3}')
print()
print(f'msg1 == msg3 hash match: {h1 == h3}  (same input → same hash)')
print(f'msg1 == msg2 hash match: {h1 == h2}  (tiny change → completely different hash)')
print(f'\nHash length is always {len(h1)} hex chars = 256 bits, regardless of input size.')

### The avalanche effect

Changing **one character** in the input completely changes the hash. This is by design — you can't predict the hash from a similar input.

---
## 4. Why plain hashing is NOT enough for passwords

If two users pick the same password, their hashes are identical. An attacker with a **rainbow table** (precomputed hash → password lookup) can crack millions of passwords instantly.

In [ ]:
# Simulate a password database with plain hashing (INSECURE)
passwords = ['password123', 'letmein', 'password123', 'admin']
print('=== Insecure: plain SHA-256 hashing ===')
plain_db = {}
for i, pw in enumerate(passwords):
    h = hashlib.sha256(pw.encode()).hexdigest()
    plain_db[f'user{i+1}'] = h
    print(f'  user{i+1}: password="{pw}"  hash={h[:24]}...')

print(f'\n⚠️  user1 and user3 have IDENTICAL hashes — attacker knows they share a password!')

# Simulate rainbow table attack
rainbow_table = {hashlib.sha256(p.encode()).hexdigest(): p for p in ['password123', 'letmein', 'admin', '123456', 'qwerty']}
print('\n🔴 Rainbow table attack:')
for user, h in plain_db.items():
    cracked = rainbow_table.get(h, '???')
    print(f'  {user}: {cracked}')
print('\n💀 All passwords cracked instantly.')

### Fix: salting + slow hashing (bcrypt)

A **salt** is a random value added to each password before hashing. Even identical passwords produce different hashes. **bcrypt** is intentionally slow, making brute-force attacks impractical.

In [ ]:
# Secure: bcrypt with per-user salt
print('=== Secure: bcrypt (salted + slow) ===')
secure_db = {}
for i, pw in enumerate(passwords):
    hashed = bcrypt.hashpw(pw.encode(), bcrypt.gensalt())
    secure_db[f'user{i+1}'] = hashed
    print(f'  user{i+1}: password="{pw}"  hash={hashed.decode()[:40]}...')

print(f'\n✅ user1 and user3 have the SAME password but DIFFERENT hashes!')
print(f'   user1 hash starts with: {secure_db["user1"].decode()[:29]}')
print(f'   user3 hash starts with: {secure_db["user3"].decode()[:29]}')

# Verify a password still works
test_pw = b'password123'
print(f'\n🔑 Verifying "password123" against user1: {bcrypt.checkpw(test_pw, secure_db["user1"])}')
print(f'🔑 Verifying "wrongpass" against user1:    {bcrypt.checkpw(b"wrongpass", secure_db["user1"])}')

### Exam tip

The exam may ask: *"What is added to a password before hashing to make rainbow table attacks infeasible?"* → **A salt**.

---
## 5. Encryption at rest vs in transit

| | At rest | In transit |
|-|---------|------------|
| **What** | Data stored on disk, in databases, in blobs | Data moving over the network |
| **How** | AES-256 (symmetric) | TLS 1.2/1.3 (asymmetric handshake → symmetric data) |
| **Azure service** | Storage Service Encryption, Azure Disk Encryption, TDE (SQL) | HTTPS enforcement, TLS everywhere |
| **Who manages keys?** | Microsoft-managed by default; you can use Customer-Managed Keys (CMK) in Key Vault | Certificate in Key Vault or Azure-managed |

In Azure, encryption at rest is **on by default** for most services. The exam loves this fact.

In [ ]:
# Simulate: data at rest encrypted, stolen by attacker
print('=== Scenario: attacker steals the database file ===\n')

# Database "file" on disk (encrypted)
key = Fernet.generate_key()  # stored in Key Vault in real Azure
cipher = Fernet(key)

records = [
    'SSN: 123-45-6789, Name: Alice Johnson',
    'SSN: 987-65-4321, Name: Bob Smith',
]

encrypted_records = [cipher.encrypt(r.encode()) for r in records]

print('What the attacker sees (encrypted bytes):')
for i, er in enumerate(encrypted_records):
    print(f'  Record {i+1}: {er[:50]}...')

print('\nWhat the authorized application sees (after decryption):')
for i, er in enumerate(encrypted_records):
    print(f'  Record {i+1}: {cipher.decrypt(er).decode()}')

print('\n💡 Without the key (stored in Key Vault), the stolen file is useless.')

---
## Summary

| Concept | What to remember |
|---------|------------------|
| **Symmetric encryption** | One key, fast, used for bulk data (AES-256) |
| **Asymmetric encryption** | Two keys (public + private), solves key distribution, used for TLS handshake |
| **Hashing** | One-way, fixed size output, can't reverse it |
| **Salting** | Random value per password that defeats rainbow tables |
| **Encryption at rest** | Data on disk — AES-256, on by default in Azure |
| **Encryption in transit** | Data on the wire — TLS, enforce HTTPS |
| **Key Vault** | Where you store and manage encryption keys, secrets, certificates |

**Next lab**: [02 — Identity and Entra](../../02-identity-and-entra/)